# TT-19 — XGBoost Regressor
## Dự báo nhu cầu thuê xe đạp công cộng theo giờ để điều phối xe

Dataset: [UCI Bike Sharing Dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) — `hour.csv`

Notebook này đi qua toàn bộ pipeline: chứng minh rò rỉ, chia dữ liệu theo thời gian, mã hoá
chu kỳ, baseline naive, train XGBoost với early stopping, so sánh log1p, dò siêu tham số,
feature importance/SHAP, biểu đồ dự báo vs thực tế, phân tích lỗi, và so sánh với Random Forest
/ Gradient Boosting.

**Lưu ý:** đặt file `hour.csv` (tải từ link trên) vào thư mục `../data/` trước khi chạy.

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from features import (
    load_data, prove_leakage, add_cyclical_features,
    build_feature_matrix, time_split, naive_baseline_predict,
)

RANDOM_STATE = 42

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

## 1. Nạp dữ liệu

In [ ]:
df = load_data("../data/hour.csv")
print(df.shape)
df.head()

## 2. ⭐ Chứng minh rò rỉ (BẪY 1)
`cnt = casual + registered` đúng bằng tổng hai cột kia. Nếu giữ lại làm đặc trưng, model sẽ
đạt R² ≈ 1.0 một cách "ảo" — không học gì cả, chỉ cộng hai số. Vì vậy hai cột này PHẢI bị bỏ.

In [ ]:
r2_leak = prove_leakage(df)
print(f"R² khi giữ casual + registered: {r2_leak:.6f}")
assert r2_leak > 0.999, "Nếu R² không xấp xỉ 1.0, kiểm tra lại dữ liệu gốc"
print("=> Xác nhận rò rỉ. Sẽ bỏ casual + registered trước khi train.")

## 3. Chia dữ liệu THEO THỜI GIAN (BẪY 2)
Đây là dữ liệu chuỗi thời gian → **không được** chia ngẫu nhiên (sẽ nhìn thấy tương lai).
Chia: năm 1 → train · 9 tháng đầu năm 2 → validation · 3 tháng cuối → test.

In [ ]:
train_df, val_df, test_df = time_split(df, val_months=9, test_months=3)
print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")

## 4. EDA — cnt trung bình theo giờ / mùa / thời tiết

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

hourly = df.groupby("hr")["cnt"].mean()
axes[0].plot(hourly.index, hourly.values, marker="o")
axes[0].set_title("Cnt trung bình theo giờ")
axes[0].axvline(8, color="gray", linestyle="--", linewidth=0.8)
axes[0].axvline(17.5, color="gray", linestyle="--", linewidth=0.8)

season_names = {1: "Xuân", 2: "Hè", 3: "Thu", 4: "Đông"}
seasonal = df.groupby("season")["cnt"].mean()
axes[1].bar([season_names[s] for s in seasonal.index], seasonal.values)
axes[1].set_title("Cnt trung bình theo mùa")

weather_names = {1: "Quang đãng", 2: "Sương mù/Mây", 3: "Mưa/Tuyết nhẹ", 4: "Mưa/Tuyết nặng"}
weather = df.groupby("weathersit")["cnt"].mean()
axes[2].bar([weather_names.get(w, str(w)) for w in weather.index], weather.values)
axes[2].set_title("Cnt trung bình theo thời tiết")
axes[2].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

print("2 đỉnh nhu cầu kỳ vọng: quanh 8h (đi làm) và 17-18h (tan tầm)")

## 5. Mã hoá đặc trưng chu kỳ (sin/cos)
`hr` (0–23): giờ 23 và giờ 0 kề nhau thực tế nhưng cách xa về giá trị số → mã hoá sin/cos.
Tương tự cho `mnth` (chu kỳ 12) và `weekday` (chu kỳ 7).

In [ ]:
X_train, y_train = build_feature_matrix(train_df, drop_leak=True)
X_val, y_val = build_feature_matrix(val_df, drop_leak=True)
X_test, y_test = build_feature_matrix(test_df, drop_leak=True)
print("Đặc trưng:", list(X_train.columns))

## 6. Baseline naive: "cùng giờ tuần trước"
Baseline này thường rất mạnh với dữ liệu có tính chu kỳ mạnh — XGBoost PHẢI thắng nó
mới chứng minh được giá trị của model phức tạp hơn.

In [ ]:
baseline_val_pred = naive_baseline_predict(train_df, val_df)
baseline_test_pred = naive_baseline_predict(pd.concat([train_df, val_df]), test_df)

print(f"Baseline RMSE validation: {rmse(y_val, baseline_val_pred):.2f}")
print(f"Baseline RMSE test: {rmse(y_test, baseline_test_pred):.2f}")

## 7. Train XGBoost + early stopping (nhãn log1p)

In [ ]:
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

model_log = xgb.XGBRegressor(
    n_estimators=2000, learning_rate=0.03, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.0, reg_alpha=0.1, min_child_weight=3,
    objective="reg:squarederror",
    early_stopping_rounds=100, eval_metric="rmse",
    tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE,
)
model_log.fit(X_train, y_train_log, eval_set=[(X_val, y_val_log)], verbose=200)

n_trees = (model_log.best_iteration + 1) if model_log.best_iteration is not None else model_log.n_estimators
print("Số cây thực tế dùng:", n_trees)

pred_val_log = np.clip(np.expm1(model_log.predict(X_val)), 0, None)
pred_test_log = np.clip(np.expm1(model_log.predict(X_test)), 0, None)

## 8. So sánh có/không dùng log1p cho nhãn

In [ ]:
model_raw = xgb.XGBRegressor(
    n_estimators=2000, learning_rate=0.03, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.0, reg_alpha=0.1, min_child_weight=3,
    objective="reg:squarederror",
    early_stopping_rounds=100, eval_metric="rmse",
    tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE,
)
model_raw.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
pred_val_raw = np.clip(model_raw.predict(X_val), 0, None)
pred_test_raw = np.clip(model_raw.predict(X_test), 0, None)

rmse_log = rmse(y_val, pred_val_log)
rmse_raw = rmse(y_val, pred_val_raw)
print(f"RMSE validation — log1p: {rmse_log:.2f} | raw: {rmse_raw:.2f}")

use_log = rmse_log <= rmse_raw
final_model = model_log if use_log else model_raw
final_pred_test = pred_test_log if use_log else pred_test_raw
print("Chọn:", "log1p" if use_log else "raw")

## 9. Dò siêu tham số bằng RandomizedSearchCV

Dùng `PredefinedSplit` để CV vẫn tôn trọng thứ tự thời gian (không xáo trộn train/validation).

In [ ]:
X_search = pd.concat([X_train, X_val], ignore_index=True)
y_search_raw = pd.concat([y_train, y_val], ignore_index=True)
y_search = np.log1p(y_search_raw) if use_log else y_search_raw

test_fold = np.array([-1] * len(X_train) + [0] * len(X_val))
ps = PredefinedSplit(test_fold)

param_dist = {
    "n_estimators": [400, 600, 800, 1000],
    "learning_rate": [0.02, 0.03, 0.05, 0.08],
    "max_depth": [4, 5, 6, 7, 8],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0],
    "reg_alpha": [0.0, 0.1, 0.5, 1.0],
    "min_child_weight": [1, 3, 5, 7],
}
search = RandomizedSearchCV(
    xgb.XGBRegressor(objective="reg:squarederror", tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE),
    param_distributions=param_dist, n_iter=25, cv=ps,
    scoring="neg_root_mean_squared_error", random_state=RANDOM_STATE, n_jobs=-1,
)
search.fit(X_search, y_search)
print("Best params:", search.best_params_)

## 10. Feature importance (gain) + SHAP summary

In [ ]:
importance = pd.Series(final_model.get_booster().get_score(importance_type="gain")).sort_values(ascending=False)
importance.head(15).sort_values().plot(kind="barh", figsize=(7, 5), title="Feature importance (gain)")
plt.tight_layout()
plt.show()

In [ ]:
import shap
explainer = shap.TreeExplainer(final_model)
sample = X_test.sample(min(1000, len(X_test)), random_state=RANDOM_STATE)
shap_values = explainer.shap_values(sample)
shap.summary_plot(shap_values, sample)

## 11. ⭐ Dự báo vs thực tế — 2 tuần cuối tập test

In [ ]:
plot_df = test_df.copy().reset_index(drop=True)
plot_df["timestamp"] = pd.to_datetime(plot_df["dteday"]) + pd.to_timedelta(plot_df["hr"], unit="h")
plot_df["y_true"] = y_test.values
plot_df["y_pred"] = final_pred_test
last_2w = plot_df.sort_values("timestamp").tail(24 * 14)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(last_2w["timestamp"], last_2w["y_true"], label="Thực tế")
ax.plot(last_2w["timestamp"], last_2w["y_pred"], label="Dự báo (XGBoost)", alpha=0.85)
ax.legend()
ax.set_title("Dự báo vs Thực tế — 2 tuần cuối")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 12. Phân tích lỗi theo giờ / thời tiết

In [ ]:
plot_df["abs_err"] = (plot_df["y_true"] - plot_df["y_pred"]).abs()
err_by_hour = plot_df.groupby("hr")["abs_err"].mean()
err_by_weather = plot_df.groupby("weathersit")["abs_err"].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
err_by_hour.plot(kind="bar", ax=axes[0], title="MAE theo giờ")
err_by_weather.rename(index=weather_names).plot(kind="bar", ax=axes[1], title="MAE theo thời tiết")
plt.tight_layout()
plt.show()

print("Giờ sai nhiều nhất:")
print(err_by_hour.sort_values(ascending=False).head(5))

## 13. So sánh với Random Forest (TT-17) và Gradient Boosting (TT-18)

In [ ]:
y_train_fit = np.log1p(y_train) if use_log else y_train

rf = RandomForestRegressor(n_estimators=400, max_depth=14, n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_train, y_train_fit)
rf_pred = rf.predict(X_test)
rf_pred = np.expm1(rf_pred) if use_log else rf_pred
rf_pred = np.clip(rf_pred, 0, None)

gb = GradientBoostingRegressor(n_estimators=400, max_depth=4, learning_rate=0.05, random_state=RANDOM_STATE)
gb.fit(X_train, y_train_fit)
gb_pred = gb.predict(X_test)
gb_pred = np.expm1(gb_pred) if use_log else gb_pred
gb_pred = np.clip(gb_pred, 0, None)

comparison = pd.DataFrame({
    "model": ["Baseline naive", "Random Forest (TT-17)", "Gradient Boosting (TT-18)", "XGBoost (TT-19)"],
    "rmse_test": [
        rmse(y_test, baseline_test_pred), rmse(y_test, rf_pred),
        rmse(y_test, gb_pred), rmse(y_test, final_pred_test),
    ],
    "r2_test": [
        r2_score(y_test, baseline_test_pred), r2_score(y_test, rf_pred),
        r2_score(y_test, gb_pred), r2_score(y_test, final_pred_test),
    ],
    "mae_test": [
        mean_absolute_error(y_test, baseline_test_pred), mean_absolute_error(y_test, rf_pred),
        mean_absolute_error(y_test, gb_pred), mean_absolute_error(y_test, final_pred_test),
    ],
})
comparison

## Lưu model
Để chạy toàn bộ pipeline production (bao gồm lưu model + report ảnh vào `models/` và
`reports/`), dùng script `src/train.py`:

```bash
python src/train.py --data data/hour.csv
```

In [ ]:
final_model.save_model("../models/xgb_bike.json")
print("Đã lưu model.")